In [3]:
import uuid
import pandas as pd
import requests
import os

In [4]:
BASE_URL = os.getenv("WEATHER_API_BASE_URL")

def fetch_weather(latitude, longitude):
    url = f"{BASE_URL}/forecast"

    params = {
        "latitude":latitude,
        "longitude": longitude,
        "hourly": ["temperature_2m", "apparent_temperature", "relative_humidity_2m", "wind_speed_10m", "weather_code", 'surface_pressure'],
        "start_date": "2026-01-01",
        "end_date": "2026-04-30",

    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    return response.json()

In [17]:
weather_code_map = {
    0: "Clear",
    1: "Partly cloudy",
    2: "Partly cloudy",
    3: "Partly cloudy",
    45: "Foggy",
    48: "Foggy",
    51: "Drizzle",
    53: "Drizzle",
    55: "Drizzle",
    56: "Freezing Drizzle",
    57: "Freezing Drizzle",
    61: "Rain",
    63: "Rain",
    65: "Rain",
    66: "Freezing Rain",
    67: "Freezing Rain",
    71: "Snow",
    73: "Snow",
    75: "Snow",
    77: "Snow Grains",
    80: "Rain Showers",
    81: "Rain Showers",
    82: "Rain Showers",
    85: "Snow Showers",
    86: "Snow Showers",
}

In [11]:
def transform_hourly_weather(weather_data, location_id):
    hourly = weather_data["hourly"]

    df = pd.DataFrame(hourly)

    df["id"] = [uuid.uuid4() for _ in range(len(df))]
    df["location_id"] = location_id

    df.rename(
        columns={
            "time": "observed_at",
            "temperature_2m": "temperature",
            "apparent_temperature": "feels_like",
            "relative_humidity_2m": "humidity",
            "surface_pressure": "pressure",
            "wind_speed_10m": "wind_speed",
        },
        inplace=True,
    )

    

    wanted_columns = [
        "id",
        "location_id",
        "observed_at",
        "temperature",
        "feels_like",
        "humidity",
        "pressure",
        "wind_speed",
        "weather_code",
    ]

    existing_columns = [col for col in wanted_columns if col in df.columns]

    return df

In [12]:
row = {
        "city": "Seattle",
        "state": "WA",
        "country": "USA",
        "latitude": 47.6062,
        "longitude": -122.3321,
    }

In [13]:
weather_data = fetch_weather(row["latitude"], row["longitude"])

In [14]:
weather_df = transform_hourly_weather(
            weather_data=weather_data,
            location_id="1234",
        )

In [15]:
weather_df

,observed_at,temperature,feels_like,humidity,wind_speed,weather_code,pressure,id,location_id
0,2026-01-01T00:00,4.2,2.4,90,1.1,3,1008.7,26c3c091-06a9-465b-bc4a-535c5a245c35,1234
1,2026-01-01T01:00,2.6,0.1,88,3.9,45,1007.8,1d4cdcc0-af62-48a4-8db4-e22e13f45731,1234
2,2026-01-01T02:00,2.8,0.7,90,1.9,45,1007.1,89b31164-e381-40ed-859e-0a77d0cab66e,1234
3,2026-01-01T03:00,3.1,1.2,94,1.5,45,1006.8,0f0faafa-6b4c-4c91-94de-b1d0639b913e,1234
4,2026-01-01T04:00,2.5,0.4,91,1.8,45,1006.4,94fcea16-ec14-4dce-898f-0c1a63543484,1234
...,...,...,...,...,...,...,...,...,...
2875,2026-04-30T19:00,18.2,19.9,68,4.7,0,1012.9,02e14fd1-db33-4ebb-9503-6c362f9454bd,1234
2876,2026-04-30T20:00,19.3,20.3,65,11.0,2,1012.2,342cf7fb-978a-4fd6-80c2-6bc6459cfe12,1234
2877,2026-04-30T21:00,20.6,21.2,58,12.4,1,1011.6,ea681de0-5181-4c8b-949a-bfcfc112f092,1234
2878,2026-04-30T22:00,22.1,22.2,47,10.0,1,1011.5,c574b087-a0c7-4a67-a343-7da3ebc6d3df,1234


In [20]:
weather_df['weather_description'] = weather_df['weather_code'].apply(lambda x: weather_code_map.get(x, 'Unknown'))